In [17]:
from langchain_community.tools import tool

# LangChain Tool Creation

LangChain supports several ways to create tools:

- `@tool` decorator
  - Decorate a Python function to expose it as a LangChain tool.
  - Automatically generates tool metadata like name, description, and argument schema.

- `StructuredTool with Pydantic`
  - Create a tool manually with explicit fields.
  - Includes `name`, `description`, `args_schema`, and `func`.

- `StructuredTool With Base model`
  - Create a tool manually with explicit fields.
  - Includes `name`, `description`, `args_schema`, and `func`.





# Using @tool method

In [18]:
# Step --> 1 : Create a function with type hint & tool decorator 



@tool
def multiply(a: int, b: int) -> int:
  """Multiply 2 numbers"""  # it is highly recommended
  return a * b



result=multiply.invoke({"a":10, "b":2})
result


20

In [19]:
print(multiply.name)
print(multiply.description)
print(multiply.coroutine)
print(multiply.from_function)
print(multiply.args)

multiply
Multiply 2 numbers
None
<bound method StructuredTool.from_function of <class 'langchain_core.tools.structured.StructuredTool'>>
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


# StructuredTool with Pydantic

In [20]:
import pydantic
from pydantic import BaseModel, Field
from typing import List, Literal, Optional, Dict

from langchain_community.tools import StructuredTool

In [21]:
# define the pydantic base model class for the function input

class MultiplyClass(BaseModel):
  a : int = Field(..., description="The first number to add")
  b : int = Field(..., description="The second number to add")


# create the function
def multiply_func(a: int, b: int) -> int:
  
  return a*b

#Make the tool
multiply_tool= StructuredTool.from_function(
  func=multiply_func,
  name = 'multiply_func',
  description="multiply 2 numbers given",
  args_schema=MultiplyClass
)

#call the tool 
result = multiply_tool.invoke({'a':1, 'b':18})
print(result)
print(multiply_tool.name)
print(multiply_tool.args)
print(multiply_tool.description)

18
multiply_func
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}
multiply 2 numbers given


# Using Base model of langchain


In [22]:
from langchain_community.tools import BaseTool
from typing import Type
from pydantic import *

In [23]:
# Step --> 1: Create the the BaseModel of pydantic 

class MultiplierClass(BaseModel):
  a : int = Field(..., description="First # to pass into the tool")
  b : int = Field(..., description="Second # to pass into the tool") 

# step --> 2: Create the base tool class

class MultiplierTool(BaseTool):
  
  #Give a name and description
  name : str = "Multiplication_tool"
  description : str = "Multiplication of 2 #"
  
  #mention the arg schema
  args_schema : Type[BaseModel] = MultiplierClass

  #define the function 
  def _run(self,a: int, b: int) -> int:
    return a * b 
  
# Step --> 3: Make the tool object 
multi_tool = MultiplierTool()

#invoke the tool 
result = multi_tool.invoke({"a": 15, "b":6})
print(result)
print(multi_tool.name)
print(multi_tool.description)
print(multi_tool.args)
   

90
Multiplication_tool
Multiplication of 2 #
{'a': {'description': 'First # to pass into the tool', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second # to pass into the tool', 'title': 'B', 'type': 'integer'}}


In [24]:
# Another tool which can make smaller text 

class inputStructure(BaseModel):
  name : str = Field(..., description ="The user name ")
  age : int = Field(..., description="the age of the user", ge=0, le=100)



class ProperStringMakerClass(BaseTool):

  name : str = "ProperStringMaker"
  description : str = "Make the name proper"
  
  args_schema: type[BaseModel] = inputStructure


  def _run(self, name, age):
    return f""" Helo {" ".join([i.capitalize() for i in name.split(" ")])} your age is : {age} years ) """



 
tool = ProperStringMakerClass()
try:
  result=tool.invoke({'name': "prem kumar", 'age':100})
except Exception as ex:
  print(f"Unexpected error: {ex}")
else:
  print(result)
finally:
  print(f"""
        n*********Execution over*******""")

 Helo Prem Kumar your age is : 100 years ) 

        n*********Execution over*******


# Tool kit
> A collection of related tools --> serves similar purpose

In [25]:
# Custom tools

class inputStructPydanticClass(BaseModel):
  a : int = Field(..., description="A number to be pass")
  b : int = Field(..., description="A number to be pass") 

# multiplication
class firstToolMulti(BaseTool):
  name : str = "Multiplication"
  description: str = "Multiplication of 2 #"

  args_schema : type[BaseModel] = inputStructPydanticClass

  def _run(self, a:int, b:int)->int:
    return a*b

multiplication_tool = firstToolMulti()


#addition
class secondToolAddition(BaseTool):
  name : str = "addition"
  description: str = "Addition of 2 #"

  args_schema : type[BaseModel] = inputStructPydanticClass

  def _run(self, a:int, b:int)->int:
    return a+b
  
addition_tool = secondToolAddition()

In [26]:
#tool kit
class MathToolkit():
  def get_tools(self):
    return [addition_tool, multiplication_tool]
  
toolkit = MathToolkit()

In [27]:
toolkit.get_tools()

[secondToolAddition(), firstToolMulti()]

In [28]:
for tool in toolkit.get_tools():
  print(f"{tool.name} --> {tool.description}")

addition --> Addition of 2 #
Multiplication --> Multiplication of 2 #


# Binding tools with llm

In [59]:
# method --> 1: giving one by one

from langchain_openai import ChatOpenAI
import os
llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

llm_with_tools=llm.bind_tools([multiplication_tool, addition_tool])



In [60]:
res=llm_with_tools.invoke("multiply and addition do both for the given numbers  9000 and 400000 and return both result")
res.content

''

In [61]:
res

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 122, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DaoMSzmybQhMyOKBSG6DmwxnBmPAJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de522-f5ad-7642-ba6e-1f01d78d88ad-0', tool_calls=[{'name': 'Multiplication', 'args': {'a': 9000, 'b': 400000}, 'id': 'call_7XzbHKf3lkzVsoPOy4PRJDpf', 'type': 'tool_call'}, {'name': 'addition', 'args': {'a': 9000, 'b': 400000}, 'id': 'call_h3QnrRA5asxxW6UCWC6cLoDr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 122, 'output_tokens': 56, 'total_tokens': 178, 'input_to

In [54]:
res.tool_calls

[{'name': 'Multiplication',
  'args': {'a': 9000, 'b': 400000},
  'id': 'call_dYgXP7xvSuuml2C4AanrLzB4',
  'type': 'tool_call'},
 {'name': 'addition',
  'args': {'a': 9000, 'b': 400000},
  'id': 'call_kHbrcgLwAe87THuNrlTYQMWd',
  'type': 'tool_call'}]

In [62]:
llm_with_tools.kwargs

{'tools': [{'type': 'function',
   'function': {'name': 'Multiplication',
    'description': 'Multiplication of 2 #',
    'parameters': {'properties': {'a': {'description': 'A number to be pass',
       'type': 'integer'},
      'b': {'description': 'A number to be pass', 'type': 'integer'}},
     'required': ['a', 'b'],
     'type': 'object'}}},
  {'type': 'function',
   'function': {'name': 'addition',
    'description': 'Addition of 2 #',
    'parameters': {'properties': {'a': {'description': 'A number to be pass',
       'type': 'integer'},
      'b': {'description': 'A number to be pass', 'type': 'integer'}},
     'required': ['a', 'b'],
     'type': 'object'}}}]}

In [ ]:
"""AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 122, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DaoMSzmybQhMyOKBSG6DmwxnBmPAJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de522-f5ad-7642-ba6e-1f01d78d88ad-0', tool_calls=[{'name': 'Multiplication', 'args': {'a': 9000, 'b': 400000}, 'id': 'call_7XzbHKf3lkzVsoPOy4PRJDpf', 'type': 'tool_call'}, {'name': 'addition', 'args': {'a': 9000, 'b': 400000}, 'id': 'call_h3QnrRA5asxxW6UCWC6cLoDr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 122, 'output_tokens': 56, 'total_tokens': 178, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})"""

"content='6' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 12, 'total_tokens': 13, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DaoFVXI3DB5RbQ6M59DisRhkRPZ23', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019de51c-5eb7-7ce2-90ae-2a74d65b3f2e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 12, 'output_tokens': 1, 'total_tokens': 13, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}"

In [64]:
# Tool calls extract karo
for tool_call in res.tool_calls:
    tool_name = tool_call['name']
    tool_args = tool_call['args']
    
    if tool_name == "Multiplication":
        result = multiply.invoke(tool_args)
        print(f"Multiplication: {result}")
    
    elif tool_name == "addition":
        result = addition_tool.invoke(tool_args)
        print(f"Addition: {result}")

Multiplication: 3600000000
Addition: 409000
